In [1]:
#### 10 splits ####

import numpy as np
import pandas as pd
from sklearn import datasets
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegressionCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
)

# ---------------------------------------------------------
# 1. Load Data
# ---------------------------------------------------------
cancer_dataset = datasets.load_breast_cancer()
X = cancer_dataset.data
y = cancer_dataset.target
feature_names = cancer_dataset.feature_names

# ---------------------------------------------------------
# 2. Settings & Models
# ---------------------------------------------------------
n_splits = 10
random_state = 42

models = {
    'k-NN (k=99)': KNeighborsClassifier(n_neighbors=99),

    'DecisionTree (max_depth=3)': Pipeline([
        ('scaler', StandardScaler()),
        ('dt', DecisionTreeClassifier(max_depth=3, random_state=random_state))
    ]),

    'SVM (linear, C=1.0)': SVC(kernel='linear', C=1.0, probability=False)
}

skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

# ---------------------------------------------------------
# 3. Cross-Validation with Elastic Net Feature Selection
# ---------------------------------------------------------
print("=== Evaluating Models with Per-Fold Elastic Net Feature Selection ===")

total_feature_selections = np.zeros(len(feature_names))

for model_name, model in models.items():
    print(f"\n============================================================")
    print(f"Evaluating Model: {model_name}")
    print(f"============================================================")

    train_accs, test_accs = [], []
    train_precs, test_precs = [], []
    train_recs, test_recs = [], []
    train_f1s, test_f1s = [], []

    for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
        fold_num = fold_idx + 1

        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]

        # Scale data
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        # -------------------------------------------------
        # Elastic Net feature selection on training fold
        # -------------------------------------------------
        enet_selector = LogisticRegressionCV(
            cv=3,
            penalty='elasticnet',
            solver='saga',
            l1_ratios=[0.1, 0.5, 0.9],
            random_state=random_state,
            max_iter=10000,
            scoring='accuracy',
            n_jobs=-1
        )
        enet_selector.fit(X_train_scaled, y_train)

        selected_indices = np.where(enet_selector.coef_[0] != 0)[0]
        selected_feature_names_in_fold = feature_names[selected_indices]

        if len(selected_indices) == 0:
            print(f"  [Warning] Fold {fold_num}: Elastic Net removed all features. Using all features as fallback.")
            selected_indices = np.arange(X.shape[1])
            selected_feature_names_in_fold = feature_names

        if model_name == list(models.keys())[0]:
            total_feature_selections[selected_indices] += 1

        X_train_sel = X_train_scaled[:, selected_indices]
        X_test_sel = X_test_scaled[:, selected_indices]

        # Train model
        model.fit(X_train_sel, y_train)

        # Predict
        y_train_pred = model.predict(X_train_sel)
        y_test_pred = model.predict(X_test_sel)

        # Metrics
        train_acc = accuracy_score(y_train, y_train_pred)
        test_acc = accuracy_score(y_test, y_test_pred)

        train_prec = precision_score(y_train, y_train_pred, zero_division=0)
        test_prec = precision_score(y_test, y_test_pred, zero_division=0)

        train_rec = recall_score(y_train, y_train_pred, zero_division=0)
        test_rec = recall_score(y_test, y_test_pred, zero_division=0)

        train_f1 = f1_score(y_train, y_train_pred, zero_division=0)
        test_f1 = f1_score(y_test, y_test_pred, zero_division=0)

        train_accs.append(train_acc)
        test_accs.append(test_acc)
        train_precs.append(train_prec)
        test_precs.append(test_prec)
        train_recs.append(train_rec)
        test_recs.append(test_rec)
        train_f1s.append(train_f1)
        test_f1s.append(test_f1)

        cm_test = confusion_matrix(y_test, y_test_pred)

        print(f"\n--- Fold {fold_num} ---")
        print(f"  Total Features Selected: {len(selected_indices)} / {X.shape[1]}")
        print(f"  Selected Feature Names : {', '.join(selected_feature_names_in_fold)}")
        print(f"  Train Metrics -> Acc: {train_acc:.4f} | Prec: {train_prec:.4f} | Rec: {train_rec:.4f} | F1: {train_f1:.4f}")
        print(f"  Test Metrics  -> Acc: {test_acc:.4f} | Prec: {test_prec:.4f} | Rec: {test_rec:.4f} | F1: {test_f1:.4f}")
        print("  Test Confusion Matrix:\n", cm_test)

    print(f"\n*** Overall Summary for {model_name} ***")
    print(f"  Train Accuracy  : {np.mean(train_accs):.4f} ± {np.std(train_accs):.4f}")
    print(f"  Test Accuracy   : {np.mean(test_accs):.4f} ± {np.std(test_accs):.4f}")
    print(f"  Train Precision : {np.mean(train_precs):.4f} ± {np.std(train_precs):.4f}")
    print(f"  Test Precision  : {np.mean(test_precs):.4f} ± {np.std(test_precs):.4f}")
    print(f"  Train Recall    : {np.mean(train_recs):.4f} ± {np.std(train_recs):.4f}")
    print(f"  Test Recall     : {np.mean(test_recs):.4f} ± {np.std(test_recs):.4f}")
    print(f"  Train F1-Score  : {np.mean(train_f1s):.4f} ± {np.std(train_f1s):.4f}")
    print(f"  Test F1-Score   : {np.mean(test_f1s):.4f} ± {np.std(test_f1s):.4f}")

# ---------------------------------------------------------
# 4. Feature Stability Report
# ---------------------------------------------------------
print("\n============================================================")
print("Elastic Net Feature Stability Across Folds")
print("============================================================")
for name, count in zip(feature_names, total_feature_selections):
    if count > 0:
        print(f"{name:<25}: selected in {int(count)}/{n_splits} folds")

=== Evaluating Models with Per-Fold Elastic Net Feature Selection ===

Evaluating Model: k-NN (k=99)

--- Fold 1 ---
  Total Features Selected: 21 / 30
  Selected Feature Names : mean radius, mean texture, mean perimeter, mean area, mean concavity, mean concave points, mean fractal dimension, radius error, perimeter error, area error, compactness error, symmetry error, fractal dimension error, worst radius, worst texture, worst perimeter, worst area, worst smoothness, worst concavity, worst concave points, worst symmetry
  Train Metrics -> Acc: 0.9414 | Prec: 0.9148 | Rec: 1.0000 | F1: 0.9555
  Test Metrics  -> Acc: 0.9825 | Prec: 0.9722 | Rec: 1.0000 | F1: 0.9859
  Test Confusion Matrix:
 [[21  1]
 [ 0 35]]

--- Fold 2 ---
  Total Features Selected: 30 / 30
  Selected Feature Names : mean radius, mean texture, mean perimeter, mean area, mean smoothness, mean compactness, mean concavity, mean concave points, mean symmetry, mean fractal dimension, radius error, texture error, perimeter 

In [2]:
#### 8 splits ####

import numpy as np
import pandas as pd
from sklearn import datasets
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegressionCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
)

# ---------------------------------------------------------
# 1. Load Data
# ---------------------------------------------------------
cancer_dataset = datasets.load_breast_cancer()
X = cancer_dataset.data
y = cancer_dataset.target
feature_names = cancer_dataset.feature_names

# ---------------------------------------------------------
# 2. Settings & Models
# ---------------------------------------------------------
n_splits = 8
random_state = 42

models = {
    'k-NN (k=99)': KNeighborsClassifier(n_neighbors=99),

    'DecisionTree (max_depth=3)': Pipeline([
        ('scaler', StandardScaler()),
        ('dt', DecisionTreeClassifier(max_depth=3, random_state=random_state))
    ]),

    'SVM (linear, C=1.0)': SVC(kernel='linear', C=1.0, probability=False)
}

skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

# ---------------------------------------------------------
# 3. Cross-Validation with Elastic Net Feature Selection
# ---------------------------------------------------------
print("=== Evaluating Models with Per-Fold Elastic Net Feature Selection ===")

total_feature_selections = np.zeros(len(feature_names))

for model_name, model in models.items():
    print(f"\n============================================================")
    print(f"Evaluating Model: {model_name}")
    print(f"============================================================")

    train_accs, test_accs = [], []
    train_precs, test_precs = [], []
    train_recs, test_recs = [], []
    train_f1s, test_f1s = [], []

    for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
        fold_num = fold_idx + 1

        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]

        # Scale data
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        # -------------------------------------------------
        # Elastic Net feature selection on training fold
        # -------------------------------------------------
        enet_selector = LogisticRegressionCV(
            cv=3,
            penalty='elasticnet',
            solver='saga',
            l1_ratios=[0.1, 0.5, 0.9],
            random_state=random_state,
            max_iter=10000,
            scoring='accuracy',
            n_jobs=-1
        )
        enet_selector.fit(X_train_scaled, y_train)

        selected_indices = np.where(enet_selector.coef_[0] != 0)[0]
        selected_feature_names_in_fold = feature_names[selected_indices]

        if len(selected_indices) == 0:
            print(f"  [Warning] Fold {fold_num}: Elastic Net removed all features. Using all features as fallback.")
            selected_indices = np.arange(X.shape[1])
            selected_feature_names_in_fold = feature_names

        if model_name == list(models.keys())[0]:
            total_feature_selections[selected_indices] += 1

        X_train_sel = X_train_scaled[:, selected_indices]
        X_test_sel = X_test_scaled[:, selected_indices]

        # Train model
        model.fit(X_train_sel, y_train)

        # Predict
        y_train_pred = model.predict(X_train_sel)
        y_test_pred = model.predict(X_test_sel)

        # Metrics
        train_acc = accuracy_score(y_train, y_train_pred)
        test_acc = accuracy_score(y_test, y_test_pred)

        train_prec = precision_score(y_train, y_train_pred, zero_division=0)
        test_prec = precision_score(y_test, y_test_pred, zero_division=0)

        train_rec = recall_score(y_train, y_train_pred, zero_division=0)
        test_rec = recall_score(y_test, y_test_pred, zero_division=0)

        train_f1 = f1_score(y_train, y_train_pred, zero_division=0)
        test_f1 = f1_score(y_test, y_test_pred, zero_division=0)

        train_accs.append(train_acc)
        test_accs.append(test_acc)
        train_precs.append(train_prec)
        test_precs.append(test_prec)
        train_recs.append(train_rec)
        test_recs.append(test_rec)
        train_f1s.append(train_f1)
        test_f1s.append(test_f1)

        cm_test = confusion_matrix(y_test, y_test_pred)

        print(f"\n--- Fold {fold_num} ---")
        print(f"  Total Features Selected: {len(selected_indices)} / {X.shape[1]}")
        print(f"  Selected Feature Names : {', '.join(selected_feature_names_in_fold)}")
        print(f"  Train Metrics -> Acc: {train_acc:.4f} | Prec: {train_prec:.4f} | Rec: {train_rec:.4f} | F1: {train_f1:.4f}")
        print(f"  Test Metrics  -> Acc: {test_acc:.4f} | Prec: {test_prec:.4f} | Rec: {test_rec:.4f} | F1: {test_f1:.4f}")
        print("  Test Confusion Matrix:\n", cm_test)

    print(f"\n*** Overall Summary for {model_name} ***")
    print(f"  Train Accuracy  : {np.mean(train_accs):.4f} ± {np.std(train_accs):.4f}")
    print(f"  Test Accuracy   : {np.mean(test_accs):.4f} ± {np.std(test_accs):.4f}")
    print(f"  Train Precision : {np.mean(train_precs):.4f} ± {np.std(train_precs):.4f}")
    print(f"  Test Precision  : {np.mean(test_precs):.4f} ± {np.std(test_precs):.4f}")
    print(f"  Train Recall    : {np.mean(train_recs):.4f} ± {np.std(train_recs):.4f}")
    print(f"  Test Recall     : {np.mean(test_recs):.4f} ± {np.std(test_recs):.4f}")
    print(f"  Train F1-Score  : {np.mean(train_f1s):.4f} ± {np.std(train_f1s):.4f}")
    print(f"  Test F1-Score   : {np.mean(test_f1s):.4f} ± {np.std(test_f1s):.4f}")

# ---------------------------------------------------------
# 4. Feature Stability Report
# ---------------------------------------------------------
print("\n============================================================")
print("Elastic Net Feature Stability Across Folds")
print("============================================================")
for name, count in zip(feature_names, total_feature_selections):
    if count > 0:
        print(f"{name:<25}: selected in {int(count)}/{n_splits} folds")

=== Evaluating Models with Per-Fold Elastic Net Feature Selection ===

Evaluating Model: k-NN (k=99)

--- Fold 1 ---
  Total Features Selected: 22 / 30
  Selected Feature Names : mean radius, mean texture, mean perimeter, mean area, mean concavity, mean concave points, mean fractal dimension, radius error, perimeter error, area error, compactness error, symmetry error, fractal dimension error, worst radius, worst texture, worst perimeter, worst area, worst smoothness, worst concavity, worst concave points, worst symmetry, worst fractal dimension
  Train Metrics -> Acc: 0.9416 | Prec: 0.9150 | Rec: 1.0000 | F1: 0.9556
  Test Metrics  -> Acc: 0.9861 | Prec: 0.9783 | Rec: 1.0000 | F1: 0.9890
  Test Confusion Matrix:
 [[26  1]
 [ 0 45]]

--- Fold 2 ---
  Total Features Selected: 30 / 30
  Selected Feature Names : mean radius, mean texture, mean perimeter, mean area, mean smoothness, mean compactness, mean concavity, mean concave points, mean symmetry, mean fractal dimension, radius error, 

In [4]:
#### 5 split ####


import numpy as np
import pandas as pd
from sklearn import datasets
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegressionCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
)

# ---------------------------------------------------------
# 1. Load Data
# ---------------------------------------------------------
cancer_dataset = datasets.load_breast_cancer()
X = cancer_dataset.data
y = cancer_dataset.target
feature_names = cancer_dataset.feature_names

# ---------------------------------------------------------
# 2. Settings & Models
# ---------------------------------------------------------
n_splits = 5
random_state = 42

models = {
    'k-NN (k=99)': KNeighborsClassifier(n_neighbors=99),

    'DecisionTree (max_depth=3)': Pipeline([
        ('scaler', StandardScaler()),
        ('dt', DecisionTreeClassifier(max_depth=3, random_state=random_state))
    ]),

    'SVM (linear, C=1.0)': SVC(kernel='linear', C=1.0, probability=False)
}

skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

# ---------------------------------------------------------
# 3. Cross-Validation with Elastic Net Feature Selection
# ---------------------------------------------------------
print("=== Evaluating Models with Per-Fold Elastic Net Feature Selection ===")

total_feature_selections = np.zeros(len(feature_names))

for model_name, model in models.items():
    print(f"\n============================================================")
    print(f"Evaluating Model: {model_name}")
    print(f"============================================================")

    train_accs, test_accs = [], []
    train_precs, test_precs = [], []
    train_recs, test_recs = [], []
    train_f1s, test_f1s = [], []

    for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
        fold_num = fold_idx + 1

        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]

        # Scale data
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        # -------------------------------------------------
        # Elastic Net feature selection on training fold
        # -------------------------------------------------
        enet_selector = LogisticRegressionCV(
            cv=3,
            penalty='elasticnet',
            solver='saga',
            l1_ratios=[0.1, 0.5, 0.9],
            random_state=random_state,
            max_iter=10000,
            scoring='accuracy',
            n_jobs=-1
        )
        enet_selector.fit(X_train_scaled, y_train)

        selected_indices = np.where(enet_selector.coef_[0] != 0)[0]
        selected_feature_names_in_fold = feature_names[selected_indices]

        if len(selected_indices) == 0:
            print(f"  [Warning] Fold {fold_num}: Elastic Net removed all features. Using all features as fallback.")
            selected_indices = np.arange(X.shape[1])
            selected_feature_names_in_fold = feature_names

        if model_name == list(models.keys())[0]:
            total_feature_selections[selected_indices] += 1

        X_train_sel = X_train_scaled[:, selected_indices]
        X_test_sel = X_test_scaled[:, selected_indices]

        # Train model
        model.fit(X_train_sel, y_train)

        # Predict
        y_train_pred = model.predict(X_train_sel)
        y_test_pred = model.predict(X_test_sel)

        # Metrics
        train_acc = accuracy_score(y_train, y_train_pred)
        test_acc = accuracy_score(y_test, y_test_pred)

        train_prec = precision_score(y_train, y_train_pred, zero_division=0)
        test_prec = precision_score(y_test, y_test_pred, zero_division=0)

        train_rec = recall_score(y_train, y_train_pred, zero_division=0)
        test_rec = recall_score(y_test, y_test_pred, zero_division=0)

        train_f1 = f1_score(y_train, y_train_pred, zero_division=0)
        test_f1 = f1_score(y_test, y_test_pred, zero_division=0)

        train_accs.append(train_acc)
        test_accs.append(test_acc)
        train_precs.append(train_prec)
        test_precs.append(test_prec)
        train_recs.append(train_rec)
        test_recs.append(test_rec)
        train_f1s.append(train_f1)
        test_f1s.append(test_f1)

        cm_test = confusion_matrix(y_test, y_test_pred)

        print(f"\n--- Fold {fold_num} ---")
        print(f"  Total Features Selected: {len(selected_indices)} / {X.shape[1]}")
        print(f"  Selected Feature Names : {', '.join(selected_feature_names_in_fold)}")
        print(f"  Train Metrics -> Acc: {train_acc:.4f} | Prec: {train_prec:.4f} | Rec: {train_rec:.4f} | F1: {train_f1:.4f}")
        print(f"  Test Metrics  -> Acc: {test_acc:.4f} | Prec: {test_prec:.4f} | Rec: {test_rec:.4f} | F1: {test_f1:.4f}")
        print("  Test Confusion Matrix:\n", cm_test)

    print(f"\n*** Overall Summary for {model_name} ***")
    print(f"  Train Accuracy  : {np.mean(train_accs):.4f} ± {np.std(train_accs):.4f}")
    print(f"  Test Accuracy   : {np.mean(test_accs):.4f} ± {np.std(test_accs):.4f}")
    print(f"  Train Precision : {np.mean(train_precs):.4f} ± {np.std(train_precs):.4f}")
    print(f"  Test Precision  : {np.mean(test_precs):.4f} ± {np.std(test_precs):.4f}")
    print(f"  Train Recall    : {np.mean(train_recs):.4f} ± {np.std(train_recs):.4f}")
    print(f"  Test Recall     : {np.mean(test_recs):.4f} ± {np.std(test_recs):.4f}")
    print(f"  Train F1-Score  : {np.mean(train_f1s):.4f} ± {np.std(train_f1s):.4f}")
    print(f"  Test F1-Score   : {np.mean(test_f1s):.4f} ± {np.std(test_f1s):.4f}")

# ---------------------------------------------------------
# 4. Feature Stability Report
# ---------------------------------------------------------
print("\n============================================================")
print("Elastic Net Feature Stability Across Folds")
print("============================================================")
for name, count in zip(feature_names, total_feature_selections):
    if count > 0:
        print(f"{name:<25}: selected in {int(count)}/{n_splits} folds")

=== Evaluating Models with Per-Fold Elastic Net Feature Selection ===

Evaluating Model: k-NN (k=99)

--- Fold 1 ---
  Total Features Selected: 30 / 30
  Selected Feature Names : mean radius, mean texture, mean perimeter, mean area, mean smoothness, mean compactness, mean concavity, mean concave points, mean symmetry, mean fractal dimension, radius error, texture error, perimeter error, area error, smoothness error, compactness error, concavity error, concave points error, symmetry error, fractal dimension error, worst radius, worst texture, worst perimeter, worst area, worst smoothness, worst compactness, worst concavity, worst concave points, worst symmetry, worst fractal dimension
  Train Metrics -> Acc: 0.9253 | Prec: 0.8987 | Rec: 0.9930 | F1: 0.9435
  Test Metrics  -> Acc: 0.9561 | Prec: 0.9459 | Rec: 0.9859 | F1: 0.9655
  Test Confusion Matrix:
 [[39  4]
 [ 1 70]]

--- Fold 2 ---
  Total Features Selected: 28 / 30
  Selected Feature Names : mean radius, mean texture, mean perime